# Análisis espectral y topológico a partir de bases SQLite

Se analizan árboles no isomorfos mediante registros previamente almacenados en SQLite. Para un árbol $T$ de orden $n$ se consideran

$$
S(T)=-\sum_{p_i>0}p_i\log_2 p_i,\qquad p_i=\frac{\lambda_i}{2(n-1)},
$$

$$
Z_1(T)=\sum_{v\in V(T)}d(v)^2,\qquad
W(T)=\sum_{\{u,v\}\subseteq V(T)}d_T(u,v).
$$

Se estudian las relaciones entre estos invariantes, su poder discriminante y las coincidencias numéricas de entropía entre espectros distintos. La comparación de entropías utiliza `EPS_ENTROPIA`.


## 1. Configuración

Se establecen el orden principal, el intervalo de estudio y las rutas de búsqueda de las bases SQLite.


In [1]:
"""Análisis de invariantes de árboles a partir de bases SQLite."""
import os
import re
import json
import math
import csv
import random
import sqlite3
import hashlib
from pathlib import Path
from collections import Counter, defaultdict, deque
from contextlib import closing

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

VERSION_ANALISIS = "sqlite-reutilizacion-2026-11-v3"
EPS_ENTROPIA = 1e-10
EPS_ESPECTRO = 1e-10
TOL_CERO = 1e-12
NUM_VERTICES = 24
ORDEN_MIN_TENDENCIA = 2
ORDEN_MAX_TENDENCIA = NUM_VERTICES
PERMITIR_INCOMPLETAS = False
MAX_PUNTOS_GRAFICO = 50000
MAX_FILAS_PREVISUALIZACION = 10000  # Los CSV conservan todos los registros.
MOSTRAR_FIGURAS = False  # Visualización opcional en el cuaderno.
PROGRESO_CADA = 1000000
MUESTRA_SEMILLA = 2026

# Rutas de búsqueda de los archivos arboles_orden_n.sqlite.
CARPETAS_BUSQUEDA = [Path(r"C:\Users\arnol\Downloads\resultados_seminario_orden_n=24")]
CARPETA_ANALISIS = Path.cwd() / "analisis_desde_sqlite"
CARPETA_CACHE = CARPETA_ANALISIS / "cache"
CARPETA_TABLAS = CARPETA_ANALISIS / f"orden_{NUM_VERTICES}" / "tablas"
CARPETA_IMAGENES = CARPETA_ANALISIS / f"orden_{NUM_VERTICES}" / "imagenes"

## 2. Lectura y validación de datos

Las bases SQLite se consultan en modo lectura. Se verifica su estructura, cardinalidad y correspondencia con el orden del árbol. Los espectros se reconstruyen únicamente para las comparaciones necesarias.


In [2]:
def asegurar_carpetas():
    for carpeta in (CARPETA_ANALISIS, CARPETA_CACHE, CARPETA_TABLAS, CARPETA_IMAGENES):
        Path(carpeta).mkdir(parents=True, exist_ok=True)


def _q(nombre):
    return '"' + str(nombre).replace('"', '""') + '"'


def buscar_bases(carpetas=CARPETAS_BUSQUEDA):
    """Localiza bases SQLite y detecta órdenes duplicados."""
    encontrados = defaultdict(list)
    visto = set()
    for raiz in carpetas:
        raiz = Path(raiz).expanduser()
        if raiz.is_file() and re.fullmatch(r"arboles_orden_(\d+)\.sqlite", raiz.name, re.I):
            rutas = [raiz]
        elif raiz.is_dir():
            rutas = raiz.rglob("arboles_orden_*.sqlite")
        else:
            print(f"No se encuentra la ruta de búsqueda: {raiz}")
            continue
        for ruta in rutas:
            ruta = ruta.resolve()
            if ruta in visto:
                continue
            visto.add(ruta)
            m = re.fullmatch(r"arboles_orden_(\d+)\.sqlite", ruta.name, re.I)
            if m:
                encontrados[int(m.group(1))].append(ruta)
    return dict(sorted(encontrados.items()))


def elegir_bases(candidatas):
    elegidas = {}
    for n, rutas in candidatas.items():
        if len(rutas) == 1:
            elegidas[n] = rutas[0]
        else:
            print(f"Bases duplicadas para n={n}; orden excluido:")
            for r in rutas:
                print("  -", r)
            print("Se requiere una base única por orden.")
    return elegidas


def abrir_base_lectura(ruta):
    ruta = Path(ruta).expanduser().resolve()
    if not ruta.is_file():
        raise FileNotFoundError(ruta)
    # Acceso a la base original en modo lectura.
    con = sqlite3.connect(ruta.as_uri() + "?mode=ro", uri=True, timeout=30)
    con.execute("PRAGMA query_only=ON")
    con.execute("PRAGMA temp_store=FILE")
    con.execute("PRAGMA cache_size=-32768")
    return con


_ALIASES = {
    "indice": ("indice", "idx", "id", "numero"),
    "codigo": ("codigo", "graph6", "codigo_graph6", "g6"),
    "entropia": ("entropia", "entropy", "entropia_vn", "vnge"),
    "zagreb1": ("zagreb1", "z1", "indice_zagreb", "primer_zagreb"),
    "wiener": ("wiener", "w", "indice_wiener"),
}


def detectar_tabla(con):
    """Identifica la tabla mediante el esquema requerido."""
    tablas = con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    ).fetchall()
    opciones = []
    for (tabla,) in tablas:
        info = con.execute(f"PRAGMA table_info({_q(tabla)})").fetchall()
        cols = {row[1].lower(): row[1] for row in info}
        campos = {}
        for canonico, alternativas in _ALIASES.items():
            coinc = next((cols[alt] for alt in alternativas if alt in cols), None)
            if coinc is None:
                break
            campos[canonico] = coinc
        if len(campos) == len(_ALIASES):
            opciones.append((tabla, campos))
    if len(opciones) != 1:
        raise ValueError(
            f"Se esperaba exactamente una tabla con las cinco columnas requeridas; "
            f"se encontraron {len(opciones)}. Tablas: {[r[0] for r in tablas]}"
        )
    return opciones[0]


def vista(con):
    tabla, campos = detectar_tabla(con)
    alias = ", ".join(f"{_q(v)} AS {_q(k)}" for k, v in campos.items())
    return f"(SELECT {alias} FROM {_q(tabla)})"


def verificar_base(n, ruta, permitir_incompletas=PERMITIR_INCOMPLETAS):
    """Verifica esquema, cardinalidad y orden de los datos."""
    with closing(abrir_base_lectura(ruta)) as con:
        v = vista(con)
        (cantidad, mn, mx, nulos) = con.execute(
            f"SELECT COUNT(*), MIN(indice), MAX(indice), "
            f"SUM(CASE WHEN codigo IS NULL OR entropia IS NULL "
            f"OR zagreb1 IS NULL OR wiener IS NULL THEN 1 ELSE 0 END) FROM {v}"
        ).fetchone()
        esperado = nx.number_of_nonisomorphic_trees(n)
        if cantidad == 0 or int(nulos or 0):
            raise ValueError(f"Base vacía o con registros nulos: {ruta}")
        if cantidad != esperado or mx - mn + 1 != cantidad or mn not in (0, 1):
            mensaje = (f"n={n}: base incompleta/incoherente: {cantidad:,} filas, "
                       f"esperadas {esperado:,}, índices {mn}..{mx}.")
            if not permitir_incompletas:
                raise ValueError(mensaje)
            print("Aviso:", mensaje, "Análisis parcial.")
        # Verificación del orden mediante un registro graph6.
        codigo = con.execute(f"SELECT codigo FROM {v} LIMIT 1").fetchone()[0]
        codigo = codigo.decode("ascii") if isinstance(codigo, bytes) else str(codigo)
        T = nx.from_graph6_bytes(codigo.encode("ascii"))
        if T.number_of_nodes() != n or not nx.is_tree(T):
            raise ValueError(f"El primer graph6 no es un árbol de orden {n}: {ruta}")
    return {"n": n, "n_arboles": cantidad, "esperado": esperado,
            "completa": cantidad == esperado, "ruta": str(Path(ruta).resolve())}


def firma_base(n, ruta):
    # Se incluyen los archivos auxiliares del registro WAL.
    archivos = []
    for archivo in (Path(ruta), Path(str(ruta) + "-wal")):
        if archivo.exists():
            st = archivo.stat()
            archivos.append([str(archivo.resolve()), st.st_size, st.st_mtime_ns])
    texto = json.dumps({"version": VERSION_ANALISIS, "n": n,
                        "archivos": archivos,
                        "eps_entropia": EPS_ENTROPIA,
                        "eps_espectro": EPS_ESPECTRO}, sort_keys=True)
    return hashlib.sha256(texto.encode()).hexdigest()


def archivo_cache(n):
    return Path(CARPETA_CACHE) / f"orden_{n}_resumen.json"


def cargar_cache(n, ruta, detallado=False):
    p = archivo_cache(n)
    if not p.exists():
        return None
    try:
        d = json.loads(p.read_text(encoding="utf8"))
        if d.get("firma") != firma_base(n, ruta):
            return None
        if detallado:
            archivos = d.get("archivos_detalle", {})
            if not archivos or not all(Path(p).exists() for p in archivos.values()):
                return None
        return d
    except (ValueError, OSError, KeyError):
        return None


def guardar_cache(n, ruta, resumen, archivos_detalle=None):
    asegurar_carpetas()
    d = {"firma": firma_base(n, ruta), "resumen": resumen,
         "archivos_detalle": archivos_detalle or {}}
    p = archivo_cache(n)
    temp = p.with_suffix(".tmp")
    temp.write_text(json.dumps(d, ensure_ascii=False, indent=2, allow_nan=True), encoding="utf8")
    temp.replace(p)


def espectro_desde_codigo(codigo):
    codigo = codigo.decode("ascii") if isinstance(codigo, bytes) else str(codigo)
    T = nx.from_graph6_bytes(codigo.encode("ascii"))
    n = T.number_of_nodes()
    L = np.zeros((n, n), dtype=float)
    for v, grado in T.degree():
        L[v, v] = grado
    for u, v in T.edges():
        L[u, v] = L[v, u] = -1.0
    vals = np.linalg.eigvalsh(L)
    vals[np.abs(vals) < TOL_CERO] = 0.0
    return vals


def agrupar_espectros_clase(grupo):
    """Agrupa espectros con Zagreb y Wiener fijos; registra multiplicidades."""
    zw = defaultdict(list)
    for r in grupo:
        zw[(r["zagreb1"], r["wiener"])].append(r)
    reps = []
    for (z, w), registros in zw.items():
        if len(registros) == 1:
            r = dict(registros[0])
            r["multiplicidad_espectral"] = 1
            reps.append(r)
            continue
        clases = []  # Espectro, representante y multiplicidad.
        for r in registros:
            ev = espectro_desde_codigo(r["codigo"])
            for item in clases:
                if np.allclose(ev, item[0], atol=EPS_ESPECTRO, rtol=0):
                    item[2] += 1
                    break
            else:
                clases.append([ev, r, 1])
        for ev, r, cantidad in clases:
            rep = dict(r)
            rep["multiplicidad_espectral"] = cantidad
            reps.append(rep)
    return reps


def elegir_muestra_reservorio(muestra, row, total, maximo, rng):
    if maximo <= 0:
        return
    if len(muestra) < maximo:
        muestra.append({k: row[k] for k in ("indice", "entropia", "zagreb1", "wiener")})
    else:
        j = rng.randrange(total)
        if j < maximo:
            muestra[j] = {k: row[k] for k in ("indice", "entropia", "zagreb1", "wiener")}


def _weighted_pearson(pairs, zx, wy):
    n = sum(c for z, w, c in pairs)
    if n < 2:
        return float('nan')
    sx = sum(c * zx(z) for z, w, c in pairs)
    sy = sum(c * wy(w) for z, w, c in pairs)
    sxx = sum(c * zx(z)**2 for z, w, c in pairs)
    syy = sum(c * wy(w)**2 for z, w, c in pairs)
    sxy = sum(c * zx(z)*wy(w) for z, w, c in pairs)
    vx = n*sxx - sx*sx
    vy = n*syy - sy*sy
    return (n*sxy - sx*sy) / math.sqrt(vx*vy) if vx > 0 and vy > 0 else float('nan')


def _rangos_promedio(conteos):
    pos = 0
    res = {}
    for k, c in sorted(conteos.items()):
        res[k] = pos + (c+1)/2
        pos += c
    return res


def estadisticas_zagreb_wiener(con, v):
    """Obtiene frecuencias conjuntas de Zagreb y Wiener desde SQLite."""
    pares = []
    z_count, w_count = Counter(), Counter()
    z_cond, w_cond = {}, {}
    iguales_zw = 0
    for z, w, c in con.execute(
        f"SELECT zagreb1, wiener, COUNT(*) FROM {v} "
        "GROUP BY zagreb1, wiener ORDER BY zagreb1, wiener"
    ):
        z,w,c = int(z),int(w),int(c)
        pares.append((z,w,c))
        z_count[z] += c
        w_count[w] += c
        iguales_zw += c*(c-1)//2
        if z not in z_cond: z_cond[z] = [w,w,set()]
        z_cond[z][0] = min(z_cond[z][0],w)
        z_cond[z][1] = max(z_cond[z][1],w)
        z_cond[z][2].add(w)
        if w not in w_cond: w_cond[w] = [z,z,set()]
        w_cond[w][0] = min(w_cond[w][0],z)
        w_cond[w][1] = max(w_cond[w][1],z)
        w_cond[w][2].add(z)
    N = sum(z_count.values())
    iz = sum(c*(c-1)//2 for c in z_count.values())
    iw = sum(c*(c-1)//2 for c in w_count.values())
    rz, rw = _rangos_promedio(z_count), _rangos_promedio(w_count)
    pearson = _weighted_pearson(pares, lambda z: z, lambda w:w)
    spearman = _weighted_pearson(pares, rz.__getitem__, rw.__getitem__)
    return {"N":N, "pares":pares, "z_count":z_count, "w_count":w_count,
            "z_cond":z_cond, "w_cond":w_cond,
            "n_clases_Z1":len(z_count), "n_clases_W":len(w_count),
            "n_clases_Z1_W":len(pares),
            "pares_mismo_Z1":iz, "pares_mismo_W":iw,
            "pares_mismo_Z1_W":iguales_zw,
            "correlacion_pearson_Z1_W":pearson,
            "correlacion_spearman_Z1_W":spearman,
            "pct_colisiones_Z1_resueltas_por_W": 100*(iz-iguales_zw)/iz if iz else float('nan'),
            "pct_colisiones_W_resueltas_por_Z1": 100*(iw-iguales_zw)/iw if iw else float('nan')}


COLUMNAS_ENTROPIA_FIJA = [
    "clase_entropia", "n_arboles", "entropia_representativa", "entropia_min", "entropia_max",
    "rango_entropia", "n_espectros_distintos", "pares_totales", "pares_cospectrales",
    "pares_no_cospectrales", "zagreb_min", "zagreb_max", "rango_zagreb",
    "n_zagreb_distintos", "wiener_min", "wiener_max", "rango_wiener",
    "n_wiener_distintos", "n_valores_Z1_W_distintos", "pares_separados_Z1",
    "porcentaje_pares_separados_Z1", "pares_separados_W", "porcentaje_pares_separados_W",
    "pares_separados_Z1_W", "porcentaje_pares_separados_Z1_W"
]

COLUMNAS_CE = [
    "clase_entropia","n_arboles","n_espectros_distintos","entropia_representativa",
    "entropia_min","entropia_max","rango_entropia","zagreb_min","zagreb_max",
    "rango_zagreb","n_zagreb_distintos","wiener_min","wiener_max","rango_wiener",
    "n_wiener_distintos","n_valores_Z1_W_distintos","D_Z1_espectral","D_W_espectral",
    "D_Z1_W_espectral","perdida_espectral_por_entropia","recuperacion_perdida_Z1",
    "recuperacion_perdida_W","recuperacion_perdida_Z1_W","pares_espectrales",
    "pares_espectrales_mismo_Z1","pares_espectrales_mismo_W",
    "pares_espectrales_mismo_Z1_W","pares_espectrales_separados_Z1",
    "pct_pares_espectrales_separados_Z1","pares_espectrales_separados_W",
    "pct_pares_espectrales_separados_W","pares_espectrales_separados_Z1_W",
    "pct_pares_espectrales_separados_Z1_W","correlacion_pearson_Z1_W"
]
COLUMNAS_CE_REP = ["clase_entropia","indice","codigo","entropia","zagreb1","wiener","multiplicidad_espectral"]
COLUMNAS_CE_COLISION = ["clase_entropia","indice1","indice2","codigo1","codigo2","entropia1","entropia2","zagreb1","wiener"]


def _crear_csv(ruta, columnas):
    f = open(ruta, 'w', newline='', encoding='utf-8-sig')
    writer = csv.DictWriter(f, fieldnames=columnas, extrasaction='ignore')
    writer.writeheader()
    return f, writer


def _resumen_grupo_espectral(grupo, reps):
    m = len(grupo)
    q = len(reps)
    zs = Counter(r['zagreb1'] for r in grupo)
    ws = Counter(r['wiener'] for r in grupo)
    zws = Counter((r['zagreb1'],r['wiener']) for r in grupo)
    total_pares = m*(m-1)//2
    cospec = sum(int(r['multiplicidad_espectral'])*(int(r['multiplicidad_espectral'])-1)//2 for r in reps)
    zsep = total_pares - sum(c*(c-1)//2 for c in zs.values())
    wsep = total_pares - sum(c*(c-1)//2 for c in ws.values())
    zwsep = total_pares - sum(c*(c-1)//2 for c in zws.values())
    Ss = [r['entropia'] for r in grupo]
    return {
        "n_arboles":m, "entropia_representativa":math.fsum(Ss)/m,
        "entropia_min":min(Ss),"entropia_max":max(Ss),"rango_entropia":max(Ss)-min(Ss),
        "n_espectros_distintos":q,"pares_totales":total_pares,
        "pares_cospectrales":cospec,"pares_no_cospectrales":total_pares-cospec,
        "zagreb_min":min(zs),"zagreb_max":max(zs),"rango_zagreb":max(zs)-min(zs),
        "n_zagreb_distintos":len(zs),"wiener_min":min(ws),"wiener_max":max(ws),
        "rango_wiener":max(ws)-min(ws),"n_wiener_distintos":len(ws),
        "n_valores_Z1_W_distintos":len(zws),
        "pares_separados_Z1":zsep,
        "porcentaje_pares_separados_Z1":100*zsep/total_pares if total_pares else float('nan'),
        "pares_separados_W":wsep,
        "porcentaje_pares_separados_W":100*wsep/total_pares if total_pares else float('nan'),
        "pares_separados_Z1_W":zwsep,
        "porcentaje_pares_separados_Z1_W":100*zwsep/total_pares if total_pares else float('nan')
    }


def _fila_CE(grupo, reps, codigo_clase, base):
    q = len(reps)
    z = Counter(r['zagreb1'] for r in reps)
    w = Counter(r['wiener'] for r in reps)
    zw = Counter((r['zagreb1'],r['wiener']) for r in reps)
    total = q*(q-1)//2
    mismos_z = sum(c*(c-1)//2 for c in z.values())
    mismos_w = sum(c*(c-1)//2 for c in w.values())
    mismos_zw = sum(c*(c-1)//2 for c in zw.values())
    perdida = q-1
    correl = _weighted_pearson([(r['zagreb1'],r['wiener'],1) for r in reps],
                               lambda x:x, lambda x:x)
    return {
        "clase_entropia":codigo_clase, "n_arboles":len(grupo),
        "n_espectros_distintos":q,
        "entropia_representativa":base['entropia_representativa'],
        "entropia_min":base['entropia_min'],"entropia_max":base['entropia_max'],
        "rango_entropia":base['rango_entropia'],
        "zagreb_min":min(z),"zagreb_max":max(z),"rango_zagreb":max(z)-min(z),
        "n_zagreb_distintos":len(z),"wiener_min":min(w),"wiener_max":max(w),
        "rango_wiener":max(w)-min(w),"n_wiener_distintos":len(w),
        "n_valores_Z1_W_distintos":len(zw),
        "D_Z1_espectral":len(z)/q,"D_W_espectral":len(w)/q,
        "D_Z1_W_espectral":len(zw)/q,
        "perdida_espectral_por_entropia":perdida,
        "recuperacion_perdida_Z1":(len(z)-1)/perdida,
        "recuperacion_perdida_W":(len(w)-1)/perdida,
        "recuperacion_perdida_Z1_W":(len(zw)-1)/perdida,
        "pares_espectrales":total,
        "pares_espectrales_mismo_Z1":mismos_z,
        "pares_espectrales_mismo_W":mismos_w,
        "pares_espectrales_mismo_Z1_W":mismos_zw,
        "pares_espectrales_separados_Z1":total-mismos_z,
        "pct_pares_espectrales_separados_Z1":100*(total-mismos_z)/total,
        "pares_espectrales_separados_W":total-mismos_w,
        "pct_pares_espectrales_separados_W":100*(total-mismos_w)/total,
        "pares_espectrales_separados_Z1_W":total-mismos_zw,
        "pct_pares_espectrales_separados_Z1_W":100*(total-mismos_zw)/total,
        "correlacion_pearson_Z1_W":correl
    }


def procesar_entropias(con, v, n, detalle=False, guardar_muestra=False):
    """Construye clases numéricas de entropía con amplitud <= EPS_ENTROPIA."""
    E=Q=E_Z=E_W=E_ZW=0
    ce = {"CE_n_clases":0,"CE_n_arboles_implicados":0,"CE_n_espectros_implicados":0,
          "CE_perdida_espectral_entropia":0,
          "CE_n_clases_Z1":0,"CE_n_clases_W":0,"CE_n_clases_Z1_W":0,
          "CE_pares_espectrales":0,"CE_pares_separados_Z1":0,
          "CE_pares_separados_W":0,"CE_pares_separados_Z1_W":0}
    # Conteos condicionados a una partición global de entropía.
    fijos_z = defaultdict(lambda: {"E":0,"Q":0,"pares_clase":0,"cospectrales":0})
    fijos_w = defaultdict(lambda: {"E":0,"Q":0,"pares_clase":0,"cospectrales":0})
    muestra=[]
    rng = random.Random(MUESTRA_SEMILLA)
    procesados = 0
    grupo=[]
    minimo=None
    det_files={}
    abiertos=[]
    writers={}
    if detalle:
        asegurar_carpetas()
        especificaciones = {
            "Entropia_fija": COLUMNAS_ENTROPIA_FIJA,
            "CE_no_cospectrales": COLUMNAS_CE,
            "CE_representantes_espectrales": COLUMNAS_CE_REP,
            "CE_colisiones_Z1_W": COLUMNAS_CE_COLISION,
        }
        for nombre, columnas in especificaciones.items():
            ruta = Path(CARPETA_TABLAS) / (nombre + ".csv")
            handle, w = _crear_csv(ruta, columnas)
            abiertos.append(handle)
            writers[nombre]=w
            det_files[nombre]=str(ruta.resolve())

    def cerrar_grupo(g):
        nonlocal E,Q,E_Z,E_W,E_ZW
        if not g: return
        E += 1
        # Reconstrucción espectral para coincidencias de (Z1, W).
        reps = agrupar_espectros_clase(g) if len(g)>1 else [dict(g[0],multiplicidad_espectral=1)]
        q = len(reps)
        Q += q
        z = Counter(r['zagreb1'] for r in g)
        w = Counter(r['wiener'] for r in g)
        zw = Counter((r['zagreb1'],r['wiener']) for r in g)
        E_Z += len(z)
        E_W += len(w)
        E_ZW += len(zw)
        zrep = defaultdict(list)
        wrep = defaultdict(list)
        for r in reps:
            zrep[r['zagreb1']].append(r)
            wrep[r['wiener']].append(r)
        for zval,c in z.items():
            nco = sum(int(r['multiplicidad_espectral'])*(int(r['multiplicidad_espectral'])-1)//2 for r in zrep[zval])
            state=fijos_z[zval]
            state['E']+=1
            state['Q']+=len(zrep[zval])
            state['pares_clase']+=c*(c-1)//2
            state['cospectrales']+=nco
        for wval,c in w.items():
            nco = sum(int(r['multiplicidad_espectral'])*(int(r['multiplicidad_espectral'])-1)//2 for r in wrep[wval])
            state=fijos_w[wval]
            state['E']+=1
            state['Q']+=len(wrep[wval])
            state['pares_clase']+=c*(c-1)//2
            state['cospectrales']+=nco
        if len(g)<2: return
        base = _resumen_grupo_espectral(g,reps)
        if detalle:
            writers['Entropia_fija'].writerow(dict(base,clase_entropia=E))
        if q<2: return
        fila = _fila_CE(g,reps,E,base)
        ce['CE_n_clases']+=1
        ce['CE_n_arboles_implicados']+=len(g)
        ce['CE_n_espectros_implicados']+=q
        ce['CE_perdida_espectral_entropia']+=q-1
        ce['CE_n_clases_Z1']+=fila['n_zagreb_distintos']
        ce['CE_n_clases_W']+=fila['n_wiener_distintos']
        ce['CE_n_clases_Z1_W']+=fila['n_valores_Z1_W_distintos']
        ce['CE_pares_espectrales']+=fila['pares_espectrales']
        ce['CE_pares_separados_Z1']+=fila['pares_espectrales_separados_Z1']
        ce['CE_pares_separados_W']+=fila['pares_espectrales_separados_W']
        ce['CE_pares_separados_Z1_W']+=fila['pares_espectrales_separados_Z1_W']
        if detalle:
            writers['CE_no_cospectrales'].writerow(fila)
            for r in reps:
                writers['CE_representantes_espectrales'].writerow(dict(r,clase_entropia=E))
            mismos=defaultdict(list)
            for r in reps:
                mismos[(r['zagreb1'],r['wiener'])].append(r)
            for (zv,wv), lista in mismos.items():
                if len(lista)<2: continue
                for i in range(len(lista)):
                    for j in range(i+1,len(lista)):
                        a,b=lista[i],lista[j]
                        writers['CE_colisiones_Z1_W'].writerow({
                            'clase_entropia':E,'indice1':a['indice'],'indice2':b['indice'],
                            'codigo1':a['codigo'],'codigo2':b['codigo'],
                            'entropia1':a['entropia'],'entropia2':b['entropia'],
                            'zagreb1':zv,'wiener':wv
                        })

    try:
        sql = f"SELECT indice, codigo, entropia, zagreb1, wiener FROM {v} ORDER BY entropia, indice"
        for indice,codigo,S,z,w in con.execute(sql):
            codigo = codigo.decode("ascii") if isinstance(codigo, bytes) else str(codigo)
            r={"indice":int(indice),"codigo":codigo,"entropia":float(S),
               "zagreb1":int(z),"wiener":int(w)}
            if not math.isfinite(r['entropia']):
                raise ValueError(f"Entropía no finita en índice {indice} de n={n}")
            procesados += 1
            if detalle and guardar_muestra:
                elegir_muestra_reservorio(muestra,r,procesados,MAX_PUNTOS_GRAFICO,rng)
            if not grupo:
                grupo=[r]
                minimo=r['entropia']
            elif r['entropia'] - minimo <= EPS_ENTROPIA:
                grupo.append(r)
            else:
                cerrar_grupo(grupo)
                grupo=[r]
                minimo=r['entropia']
            if PROGRESO_CADA and procesados%PROGRESO_CADA==0:
                print(f"  n={n}: {procesados:,} registros leídos y agrupados...")
        cerrar_grupo(grupo)
    finally:
        for h in abiertos: h.close()
    C=ce['CE_n_clases'];Qt=ce['CE_n_espectros_implicados'];per=ce['CE_perdida_espectral_entropia']
    p=ce['CE_pares_espectrales']
    ce.update({
       'CE_D_Z1_espectral':ce['CE_n_clases_Z1']/Qt if Qt else float('nan'),
       'CE_D_W_espectral':ce['CE_n_clases_W']/Qt if Qt else float('nan'),
       'CE_D_Z1_W_espectral':ce['CE_n_clases_Z1_W']/Qt if Qt else float('nan'),
       'CE_recuperacion_perdida_Z1':(ce['CE_n_clases_Z1']-C)/per if per else float('nan'),
       'CE_recuperacion_perdida_W':(ce['CE_n_clases_W']-C)/per if per else float('nan'),
       'CE_recuperacion_perdida_Z1_W':(ce['CE_n_clases_Z1_W']-C)/per if per else float('nan'),
       'CE_pct_pares_separados_Z1':100*ce['CE_pares_separados_Z1']/p if p else float('nan'),
       'CE_pct_pares_separados_W':100*ce['CE_pares_separados_W']/p if p else float('nan'),
       'CE_pct_pares_separados_Z1_W':100*ce['CE_pares_separados_Z1_W']/p if p else float('nan')
    })
    if detalle and guardar_muestra:
        muestra_df=pd.DataFrame(muestra)
        ruta=Path(CARPETA_TABLAS)/'Muestra_graficos.csv'
        muestra_df.to_csv(ruta,index=False,encoding='utf-8-sig')
        det_files['Muestra_graficos']=str(ruta.resolve())
    return {"E":E,"Q":Q,"E_Z":E_Z,"E_W":E_W,"E_ZW":E_ZW,
            "fijos_z":dict(fijos_z),"fijos_w":dict(fijos_w),
            "ce":ce,"procesados":procesados,"archivos":det_files}



def contar_entropias_por_valor(con, v, campo):
    """Agrupa entropías por invariante fijo y cuenta pares próximos mediante ventana móvil."""
    resultado = {}
    actual = object()
    minimo = None
    E = pares = 0
    ventana = deque()
    sql = f"SELECT {campo}, entropia FROM {v} ORDER BY {campo}, entropia"
    for valor,S in con.execute(sql):
        valor = int(valor)
        S = float(S)
        if valor != actual:
            if minimo is not None:
                resultado[actual] = {'E_independiente':E,'pares_eps':pares}
            actual = valor
            minimo = S
            E = 1
            pares = 0
            ventana.clear()
        elif S - minimo > EPS_ENTROPIA:
            minimo = S
            E += 1
        while ventana and S - ventana[0] > EPS_ENTROPIA:
            ventana.popleft()
        pares += len(ventana)
        ventana.append(S)
    if minimo is not None:
        resultado[actual] = {'E_independiente':E,'pares_eps':pares}
    return resultado


def _resumen_condicional(fixed_state, subinfo, total_por_valor, secundario, conteos_directos=None):
    filas=[]
    for valor, n_arboles in sorted(total_por_valor.items()):
        state=fixed_state.get(valor,{"E":0,"Q":0,"pares_clase":0,"cospectrales":0})
        mn, mx, distintos = subinfo[valor]
        Q=state['Q']
        E=conteos_directos[valor]['E_independiente'] if conteos_directos else state['E']
        pares_eps=conteos_directos[valor]['pares_eps'] if conteos_directos else state['pares_clase']
        filas.append({
          'valor':int(valor), 'n_arboles':int(n_arboles),
          'n_grupos_numericos_entropia':E,'n_espectros_distintos':Q,
          'poder_discriminante_entropia':E/n_arboles if n_arboles else float('nan'),
          'perdida_arbol_a_espectro':n_arboles-Q,
          'perdida_espectro_a_entropia':Q-E,
          'perdida_total_arbol_a_entropia':n_arboles-E,
          'pares_misma_clase_S':state['pares_clase'],
          'pares_eps_coentropicos':pares_eps,
          'pares_cospectrales':state['cospectrales'],
          'pares_eps_coentropicos_no_cospectrales':pares_eps-state['cospectrales'],
          'pares_misma_clase_S_no_cospectrales':state['pares_clase']-state['cospectrales'],
          f'{secundario}_min':mn,f'{secundario}_max':mx,
          f'rango_{secundario}':mx-mn,
          f'n_{secundario}_distintos':len(distintos)
        })
    return pd.DataFrame(filas)


def stats_por_valor(con, v, campo):
    return {
        int(x[0]):dict(entropia_min=float(x[1]),entropia_max=float(x[2]),
                       entropia_media=float(x[3]),
                       entropia_desviacion=math.sqrt(max(0.,float(x[4])-float(x[3])**2)),
                       rango_entropia=float(x[2])-float(x[1]))
        for x in con.execute(
            f'SELECT {campo},MIN(entropia),MAX(entropia),AVG(entropia),'
            f'AVG(entropia*entropia) FROM {v} GROUP BY {campo}'
        )
    }


def construir_resultados_orden(n, ruta, detalle=False):
    print(f"\nAnálisis n={n}: {ruta}")
    info=verificar_base(n,ruta)
    with closing(abrir_base_lectura(ruta)) as con:
        v=vista(con)
        stats=estadisticas_zagreb_wiener(con,v)
        scan=procesar_entropias(con,v,n,detalle=detalle,guardar_muestra=detalle)
        if stats['N']!=scan['procesados'] or stats['N']!=info['n_arboles']:
            raise RuntimeError('La cantidad de filas leídas no coincide; comprobar estado de la base.')
        N=stats['N'];E=scan['E'];Q=scan['Q']
        rango_z=max(stats['z_count'])-min(stats['z_count'])
        rango_w=max(stats['w_count'])-min(stats['w_count'])
        def rango_cond(d):
            rangos=[a[1]-a[0] for a in d.values()]
            return sum(rangos)/len(rangos) if rangos else 0.,max(rangos) if rangos else 0
        rzmed,rzmax=rango_cond(stats['z_cond'])
        rwmed,rwmax=rango_cond(stats['w_cond'])
        out={
          'n':n,'n_arboles':N,'base_completa':info['completa'],
          'n_clases_S':E,'n_espectros_distintos':Q,
          'n_clases_Z1':stats['n_clases_Z1'],'n_clases_W':stats['n_clases_W'],
          'n_clases_Z1_W':stats['n_clases_Z1_W'],
          'D_S':E/N,'D_Z1':stats['n_clases_Z1']/N,
          'D_W':stats['n_clases_W']/N,'D_Z1_W':stats['n_clases_Z1_W']/N,
          'D_S_Z1':scan['E_Z']/N,'D_S_W':scan['E_W']/N,'D_S_Z1_W':scan['E_ZW']/N,
          'correlacion_pearson_Z1_W':stats['correlacion_pearson_Z1_W'],
          'correlacion_spearman_Z1_W':stats['correlacion_spearman_Z1_W'],
          'pares_mismo_Z1':stats['pares_mismo_Z1'],
          'pares_mismo_W':stats['pares_mismo_W'],
          'pares_mismo_Z1_W':stats['pares_mismo_Z1_W'],
          'pct_colisiones_Z1_resueltas_por_W':stats['pct_colisiones_Z1_resueltas_por_W'],
          'pct_colisiones_W_resueltas_por_Z1':stats['pct_colisiones_W_resueltas_por_Z1'],
          'promedio_rango_W_dado_Z1':rzmed,'max_rango_W_dado_Z1':rzmax,
          'promedio_rango_Z1_dado_W':rwmed,'max_rango_Z1_dado_W':rwmax,
          'promedio_rango_relativo_W_dado_Z1':rzmed/rango_w if rango_w else 0.,
          'max_rango_relativo_W_dado_Z1':rzmax/rango_w if rango_w else 0.,
          'promedio_rango_relativo_Z1_dado_W':rwmed/rango_z if rango_z else 0.,
          'max_rango_relativo_Z1_dado_W':rwmax/rango_z if rango_z else 0.,
          'perdida_arbol_a_espectro':N-Q,
          'perdida_espectro_a_entropia':Q-E,
          'perdida_total_arbol_a_entropia':N-E,
          **scan['ce']
        }
        if detalle:
            print(f" n={n}: agrupación con Zagreb fijo...")
            z_directos = contar_entropias_por_valor(con, v, 'zagreb1')
            zdf=_resumen_condicional(scan['fijos_z'],stats['z_cond'],stats['z_count'],'wiener',z_directos)
            zdf=zdf.rename(columns={'valor':'zagreb1'}).merge(
                pd.DataFrame.from_dict(stats_por_valor(con,v,'zagreb1'),orient='index').rename_axis('zagreb1').reset_index(),on='zagreb1')
            print(f" n={n}: agrupación con Wiener fijo...")
            w_directos = contar_entropias_por_valor(con, v, 'wiener')
            wdf=_resumen_condicional(scan['fijos_w'],stats['w_cond'],stats['w_count'],'zagreb1',w_directos)
            wdf=wdf.rename(columns={'valor':'wiener'}).merge(
                pd.DataFrame.from_dict(stats_por_valor(con,v,'wiener'),orient='index').rename_axis('wiener').reset_index(),on='wiener')
            zdf.to_csv(Path(CARPETA_TABLAS)/'Zagreb_fijo.csv',index=False,encoding='utf-8-sig')
            wdf.to_csv(Path(CARPETA_TABLAS)/'Wiener_fijo.csv',index=False,encoding='utf-8-sig')
            scan['archivos']['Zagreb_fijo']=str((Path(CARPETA_TABLAS)/'Zagreb_fijo.csv').resolve())
            scan['archivos']['Wiener_fijo']=str((Path(CARPETA_TABLAS)/'Wiener_fijo.csv').resolve())
    return out,scan['archivos']


def analizar_archivos_existentes(bases, orden_min=ORDEN_MIN_TENDENCIA,
                                 orden_max=ORDEN_MAX_TENDENCIA, principal=NUM_VERTICES):
    asegurar_carpetas()
    ordenes=sorted(set(range(orden_min,orden_max+1)) | {principal})
    filas=[]
    faltantes=[]
    manifiesto=[]
    for n in ordenes:
        if n not in bases:
            print(f"n={n}: base no disponible; orden excluido.")
            faltantes.append(n)
            continue
        ruta=bases[n]
        detalle=(n==principal)
        try:
            cache=cargar_cache(n,ruta,detallado=detalle)
            if cache is not None:
                print(f"n={n}: análisis recuperado de caché.")
                resumen=cache['resumen']
            else:
                resumen, archivos=construir_resultados_orden(n,ruta,detalle=detalle)
                guardar_cache(n,ruta,resumen,archivos_detalle=archivos)
            filas.append(resumen)
            manifiesto.append({'n':n,'ruta_sqlite':str(ruta),'estado':'completa' if resumen['base_completa'] else 'parcial'})
        except Exception as exc:
            print(f"n={n}: base excluida ({type(exc).__name__}: {exc}).")
            faltantes.append(n)
            manifiesto.append({'n':n,'ruta_sqlite':str(ruta),'estado':f'error: {exc}'})
    tendencia=pd.DataFrame(filas).sort_values('n') if filas else pd.DataFrame()
    pd.DataFrame(manifiesto).to_csv(Path(CARPETA_ANALISIS)/'bases_utilizadas.csv',index=False,encoding='utf-8-sig')
    if not tendencia.empty:
        tendencia.to_csv(Path(CARPETA_ANALISIS)/'tendencia_ordenes.csv',index=False,encoding='utf-8-sig')
    print('Órdenes sin resultados:',faltantes)
    return tendencia,faltantes


def tablas_principales(tendencia):
    fila=tendencia.loc[tendencia['n']==NUM_VERTICES]
    if fila.empty:
        raise ValueError(f"No hay análisis completo disponible para n={NUM_VERTICES}.")
    r=fila.iloc[0].to_dict()
    N=r['n_arboles']; Q=r['n_espectros_distintos']; E=r['n_clases_S']
    nombres=[('Entropía S',E),('Z1',r['n_clases_Z1']),('Wiener W',r['n_clases_W']),
             ('(Z1, W)',r['n_clases_Z1_W']),
             ('(S, Z1)',r['D_S_Z1']*N),('(S, W)',r['D_S_W']*N),
             ('(S, Z1, W)',r['D_S_Z1_W']*N)]
    discriminacion=pd.DataFrame([{'descriptor':et,'n_clases':round(c),'n_arboles':N,
                                 'poder_discriminante':c/N,
                                 'porcentaje_discriminante':100*c/N} for et,c in nombres])
    perdida=pd.DataFrame([{'n':NUM_VERTICES,'n_arboles':N,'n_espectros_distintos':Q,
                           'n_grupos_numericos_entropia':E,
                           'perdida_arbol_a_espectro':N-Q,
                           'perdida_espectro_a_entropia':Q-E,
                           'perdida_total_arbol_a_entropia':N-E,
                           'retencion_espectral':Q/N,
                           'retencion_entropia_desde_arbol':E/N,
                           'retencion_entropia_desde_espectro':E/Q}])
    relaciones=pd.DataFrame([{
        'n':NUM_VERTICES, 'n_arboles':N,
        'correlacion_pearson_Z1_W':r['correlacion_pearson_Z1_W'],
        'correlacion_spearman_Z1_W':r['correlacion_spearman_Z1_W'],
        'n_valores_Z1':r['n_clases_Z1'], 'n_valores_W':r['n_clases_W'],
        'n_valores_Z1_W_distintos':r['n_clases_Z1_W'],
        'pares_totales':N*(N-1)//2,
        'pares_mismo_Z1':r['pares_mismo_Z1'],
        'pares_mismo_W':r['pares_mismo_W'],
        'pares_mismo_Z1_W':r['pares_mismo_Z1_W'],
        'colisiones_Z1_resueltas_por_W':r['pares_mismo_Z1']-r['pares_mismo_Z1_W'],
        'porcentaje_colisiones_Z1_resueltas_por_W':r['pct_colisiones_Z1_resueltas_por_W'],
        'colisiones_W_resueltas_por_Z1':r['pares_mismo_W']-r['pares_mismo_Z1_W'],
        'porcentaje_colisiones_W_resueltas_por_Z1':r['pct_colisiones_W_resueltas_por_Z1']
    }])
    ce=pd.DataFrame([{
        'n':NUM_VERTICES, 'n_clases':r['CE_n_clases'],
        'n_arboles_implicados':r['CE_n_arboles_implicados'],
        'n_espectros_implicados':r['CE_n_espectros_implicados'],
        'perdida_espectral_entropia':r['CE_perdida_espectral_entropia'],
        'recuperacion_perdida_Z1':r['CE_recuperacion_perdida_Z1'],
        'recuperacion_perdida_W':r['CE_recuperacion_perdida_W'],
        'recuperacion_perdida_Z1_W':r['CE_recuperacion_perdida_Z1_W'],
    }])
    ce_tend=tendencia[['n']+[c for c in tendencia.columns if c.startswith('CE_')]].copy()
    return discriminacion,perdida,relaciones,ce,ce_tend


def leer_tabla_resumen(nombre, max_filas=MAX_FILAS_PREVISUALIZACION):
    p=Path(CARPETA_TABLAS)/(nombre+'.csv')
    if not p.exists():
        return pd.DataFrame()
    # Lectura parcial para previsualización; CSV completo en disco.
    return pd.read_csv(p,nrows=max_filas)


def exportar_resumenes_excel(tablas):
    """Exporta tablas de resumen respetando los límites de Excel."""
    ruta=Path(CARPETA_TABLAS)/f"analisis_desde_bases_orden_{NUM_VERTICES}.xlsx"
    limite_excel=1048575  # Filas de datos sin encabezado.
    with pd.ExcelWriter(ruta,engine='openpyxl') as writer:
        for nombre,df in tablas.items():
            if df is None: continue
            if len(df)>limite_excel:
                print(f"{nombre}: tabla excluida de Excel por límite de filas; disponible en CSV.")
                continue
            df.to_excel(writer,sheet_name=nombre[:31],index=False)
    return ruta


## 3. Localización de bases SQLite

Se identifican las bases disponibles por orden y se registran sus rutas. Los órdenes con archivos duplicados se excluyen hasta resolver la ambigüedad.


In [3]:

bases_candidatas = buscar_bases(CARPETAS_BUSQUEDA)
bases_existentes = elegir_bases(bases_candidatas)
if not bases_existentes:
    raise FileNotFoundError(
        "No se localizaron archivos arboles_orden_n.sqlite en las rutas indicadas."
    )
display(pd.DataFrame([{'n':n,'ruta':str(p)} for n,p in bases_existentes.items()]))
print(f"Orden principal: n={NUM_VERTICES}")
print(f"Tendencia solicitada: n={ORDEN_MIN_TENDENCIA},...,{ORDEN_MAX_TENDENCIA}")
print("Origen: bases SQLite existentes.")


,n,ruta
0,2,C:\Users\arnol\Downloads\resultados_seminario_...
1,3,C:\Users\arnol\Downloads\resultados_seminario_...
2,4,C:\Users\arnol\Downloads\resultados_seminario_...
3,5,C:\Users\arnol\Downloads\resultados_seminario_...
4,6,C:\Users\arnol\Downloads\resultados_seminario_...
5,7,C:\Users\arnol\Downloads\resultados_seminario_...
6,8,C:\Users\arnol\Downloads\resultados_seminario_...
7,9,C:\Users\arnol\Downloads\resultados_seminario_...
8,10,C:\Users\arnol\Downloads\resultados_seminario_...
9,11,C:\Users\arnol\Downloads\resultados_seminario_...


Orden principal: n=24
Tendencia solicitada: n=2,...,24
Modo de trabajo: SOLO BASES EXISTENTES; sin enumeración.


## 4. Análisis de datos almacenados

Las consultas y agrupaciones se efectúan sobre SQLite. Se reutilizan los resúmenes en caché cuando los archivos no han cambiado. Los órdenes ausentes o con bases incompletas se excluyen de los resultados exhaustivos.


In [4]:

tendencia_df, ordenes_omitidos = analizar_archivos_existentes(
    bases_existentes,
    orden_min=ORDEN_MIN_TENDENCIA,
    orden_max=ORDEN_MAX_TENDENCIA,
    principal=NUM_VERTICES
)
if tendencia_df.empty:
    raise RuntimeError("No hubo órdenes válidos para analizar; revisa los mensajes de validación.")
display(tendencia_df.head(30))
print("Órdenes incluidos:", list(tendencia_df['n']))
if NUM_VERTICES not in set(tendencia_df['n']):
    raise RuntimeError(
        f"Datos incompletos o ausentes para el orden principal n={NUM_VERTICES}."
    )



Procesando n=2 desde C:\Users\arnol\Downloads\resultados_seminario_orden_n=24\datos\arboles_orden_2.sqlite (sin regenerar árboles)...

Procesando n=3 desde C:\Users\arnol\Downloads\resultados_seminario_orden_n=24\datos\arboles_orden_3.sqlite (sin regenerar árboles)...

Procesando n=4 desde C:\Users\arnol\Downloads\resultados_seminario_orden_n=24\datos\arboles_orden_4.sqlite (sin regenerar árboles)...

Procesando n=5 desde C:\Users\arnol\Downloads\resultados_seminario_orden_n=24\datos\arboles_orden_5.sqlite (sin regenerar árboles)...

Procesando n=6 desde C:\Users\arnol\Downloads\resultados_seminario_orden_n=24\datos\arboles_orden_6.sqlite (sin regenerar árboles)...

Procesando n=7 desde C:\Users\arnol\Downloads\resultados_seminario_orden_n=24\datos\arboles_orden_7.sqlite (sin regenerar árboles)...

Procesando n=8 desde C:\Users\arnol\Downloads\resultados_seminario_orden_n=24\datos\arboles_orden_8.sqlite (sin regenerar árboles)...

Procesando n=9 desde C:\Users\arnol\Downloads\resultad

,n,n_arboles,base_completa,n_clases_S,n_espectros_distintos,n_clases_Z1,n_clases_W,n_clases_Z1_W,D_S,D_Z1,...,CE_pares_separados_Z1_W,CE_D_Z1_espectral,CE_D_W_espectral,CE_D_Z1_W_espectral,CE_recuperacion_perdida_Z1,CE_recuperacion_perdida_W,CE_recuperacion_perdida_Z1_W,CE_pct_pares_separados_Z1,CE_pct_pares_separados_W,CE_pct_pares_separados_Z1_W
0,2,1,True,1,1,1,1,1,1.000000,1.000000,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,1,True,1,1,1,1,1,1.000000,1.000000,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,2,True,2,2,2,2,2,1.000000,1.000000,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,3,True,3,3,3,3,3,1.000000,1.000000,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,6,True,6,6,5,6,6,1.000000,0.833333,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,7,11,True,11,11,7,9,11,1.000000,0.636364,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,8,23,True,23,23,9,20,23,1.000000,0.391304,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9,47,True,47,47,13,21,40,1.000000,0.276596,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,10,106,True,106,106,18,53,94,1.000000,0.169811,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,11,235,True,232,232,21,51,146,0.987234,0.089362,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Órdenes analizados: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


## 5. Tablas de resultados

Se presentan los resúmenes del orden principal y de la evolución con $n$. Las tablas extensas se conservan íntegramente en CSV y se muestran parcialmente en el cuaderno.


In [5]:

(
    resumen_discriminacion_df,
    resumen_perdida_global_df,
    relacion_zagreb_wiener_df,
    resumen_coentropicas_no_cospectrales_df,
    tendencia_coentropicas_no_cospectrales_df
) = tablas_principales(tendencia_df)
resumen_zagreb_df = leer_tabla_resumen('Zagreb_fijo')
resumen_wiener_df = leer_tabla_resumen('Wiener_fijo')
entropia_fija_df = leer_tabla_resumen('Entropia_fija')
coentropicas_no_cospectrales_df = leer_tabla_resumen('CE_no_cospectrales')
representantes_espectrales_ce_df = leer_tabla_resumen('CE_representantes_espectrales')
colisiones_Z1_W_ce_df = leer_tabla_resumen('CE_colisiones_Z1_W')
muestra_df = leer_tabla_resumen('Muestra_graficos', max_filas=MAX_PUNTOS_GRAFICO)
resultados_muestra = muestra_df.to_dict('records')
# Los órdenes ausentes se representan mediante valores NaN.
tendencia_para_graficos = (
    tendencia_df.set_index('n')
    .reindex(range(ORDEN_MIN_TENDENCIA, ORDEN_MAX_TENDENCIA + 1))
    .rename_axis('n').reset_index()
)
tendencia_ce_para_graficos = (
    tendencia_coentropicas_no_cospectrales_df.set_index('n')
    .reindex(range(ORDEN_MIN_TENDENCIA, ORDEN_MAX_TENDENCIA + 1))
    .rename_axis('n').reset_index()
)
print('Resumen discriminante:')
display(resumen_discriminacion_df)
print('Pérdida global:')
display(resumen_perdida_global_df)
print('Relación Zagreb y Wiener:')
display(relacion_zagreb_wiener_df)
print("Resumen con Zagreb fijo:")
display(resumen_zagreb_df.head(10))
print("Resumen con Wiener fijo:")
display(resumen_wiener_df.head(10))
print("Clases coentrópicas no cospectrales:")
display(coentropicas_no_cospectrales_df.head(10))


Resumen discriminante:


,descriptor,n_clases,n_arboles,poder_discriminante,porcentaje_discriminante
0,Entropía S,38263578,39299897,0.973630,97.363049
1,Z1,145,39299897,0.000004,0.000369
2,Wiener W,1558,39299897,0.000040,0.003964
3,"(Z1, W)",48693,39299897,0.001239,0.123901
4,"(S, Z1)",38741141,39299897,0.985782,98.578225
5,"(S, W)",39294805,39299897,0.999870,99.987043
6,"(S, Z1, W)",39295761,39299897,0.999895,99.989476


Pérdida global:


,n,n_arboles,n_espectros_distintos,n_grupos_numericos_entropia,perdida_arbol_a_espectro,perdida_espectro_a_entropia,perdida_total_arbol_a_entropia,retencion_espectral,retencion_entropia_desde_arbol,retencion_entropia_desde_espectro
0,24,39299897,39298085,38263578,1812,1034507,1036319,0.999954,0.97363,0.973675


Relación Zagreb y Wiener:


,n,n_arboles,correlacion_pearson_Z1_W,correlacion_spearman_Z1_W,n_valores_Z1,n_valores_W,n_valores_Z1_W_distintos,pares_totales,pares_mismo_Z1,pares_mismo_W,pares_mismo_Z1_W,colisiones_Z1_resueltas_por_W,porcentaje_colisiones_Z1_resueltas_por_W,colisiones_W_resueltas_por_Z1,porcentaje_colisiones_W_resueltas_por_Z1
0,24,39299897,-0.650771,-0.685209,145,1558,48693,772240932455356,42517090487111,1400570671585,110893651615,42406196835496,99.739179,1289677019970,92.082252


... (2 outputs left out)

Wiener fijo, muestra:


,wiener,n_arboles,n_grupos_numericos_entropia,n_espectros_distintos,poder_discriminante_entropia,perdida_arbol_a_espectro,perdida_espectro_a_entropia,perdida_total_arbol_a_entropia,pares_misma_clase_S,pares_eps_coentropicos,...,pares_misma_clase_S_no_cospectrales,zagreb1_min,zagreb1_max,rango_zagreb1,n_zagreb1_distintos,entropia_min,entropia_max,entropia_media,entropia_desviacion,rango_entropia
0,529,1,1,1,1.0,0,0,0,0,0,...,0,552,552,0,1,3.131408,3.131408,3.131408,0.000000,0.000000
1,550,1,1,1,1.0,0,0,0,0,0,...,0,510,510,0,1,3.195325,3.195325,3.195325,0.000000,0.000000
2,569,1,1,1,1.0,0,0,0,0,0,...,0,472,472,0,1,3.249600,3.249600,3.249600,0.000000,0.000000
3,571,1,1,1,1.0,0,0,0,0,0,...,0,470,470,0,1,3.257950,3.257950,3.257950,0.000000,0.000000
4,586,1,1,1,1.0,0,0,0,0,0,...,0,438,438,0,1,3.295819,3.295819,3.295819,0.000000,0.000000
5,590,2,2,2,1.0,0,0,0,0,0,...,0,434,470,36,2,3.260758,3.310869,3.285814,0.025055,0.050111
6,592,1,1,1,1.0,0,0,0,0,0,...,0,432,432,0,1,3.319227,3.319227,3.319227,0.000000,0.000000
7,601,1,1,1,1.0,0,0,0,0,0,...,0,408,408,0,1,3.335032,3.335032,3.335032,0.000000,0.000000
8,607,2,2,2,1.0,0,0,0,0,0,...,0,402,434,32,2,3.313113,3.355668,3.334391,0.021278,0.042556
9,609,1,1,1,1.0,0,0,0,0,0,...,0,400,400,0,1,3.362364,3.362364,3.362364,0.000000,0.000000


Clases coentrópicas no cospectrales, muestra:


,clase_entropia,n_arboles,n_espectros_distintos,entropia_representativa,entropia_min,entropia_max,rango_entropia,zagreb_min,zagreb_max,rango_zagreb,...,pares_espectrales_mismo_Z1,pares_espectrales_mismo_W,pares_espectrales_mismo_Z1_W,pares_espectrales_separados_Z1,pct_pares_espectrales_separados_Z1,pares_espectrales_separados_W,pct_pares_espectrales_separados_W,pares_espectrales_separados_Z1_W,pct_pares_espectrales_separados_Z1_W,correlacion_pearson_Z1_W
0,38477,2,2,3.735245,3.735245,3.735245,7.894840e-11,186,208,22,...,0,0,0,1,100.0,1,100.0,1,100.0,1.0
1,40153,2,2,3.736840,3.736840,3.736840,3.156631e-11,196,206,10,...,0,0,0,1,100.0,1,100.0,1,100.0,-1.0
2,41385,2,2,3.737982,3.737982,3.737982,6.715739e-11,192,202,10,...,0,0,0,1,100.0,1,100.0,1,100.0,1.0
3,43031,2,2,3.739490,3.739490,3.739490,8.220757e-11,208,208,0,...,1,0,0,0,0.0,1,100.0,1,100.0,NaN
4,50584,2,2,3.745906,3.745906,3.745906,6.465495e-12,186,196,10,...,0,0,0,1,100.0,1,100.0,1,100.0,1.0
5,58059,2,2,3.751175,3.751175,3.751175,7.206413e-11,196,196,0,...,1,0,0,0,0.0,1,100.0,1,100.0,NaN
6,58838,2,2,3.751716,3.751716,3.751716,5.897771e-11,188,190,2,...,0,0,0,1,100.0,1,100.0,1,100.0,-1.0
7,59383,2,2,3.752056,3.752056,3.752056,1.038458e-11,190,196,6,...,0,0,0,1,100.0,1,100.0,1,100.0,1.0
8,62365,2,2,3.753980,3.753980,3.753980,7.465584e-11,194,194,0,...,1,0,0,0,0.0,1,100.0,1,100.0,NaN
9,64819,2,2,3.755505,3.755505,3.755505,1.610712e-11,190,190,0,...,1,0,0,0,0.0,1,100.0,1,100.0,NaN


## 6. Representación gráfica

En la comparación Zagreb–Wiener, $W$ ocupa el eje horizontal y $Z_1$ el vertical. Las dispersiones usan muestras reproducibles; los indicadores y tendencias se calculan con todos los registros válidos. Los órdenes ausentes no se interpolan.


In [6]:
def figura_entropia_vs_invariante(
    resultados,
    campo,
    etiqueta,
    titulo
):
    x = [
        r[campo]
        for r in resultados
    ]

    y = [
        r["entropia"]
        for r in resultados
    ]

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.scatter(
        x,
        y,
        s=9,
        alpha=0.35
    )

    ax.set_xlabel(etiqueta)
    ax.set_ylabel(
        "Entropía de von Neumann"
    )
    ax.set_title(titulo)

    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_wiener_vs_zagreb(
    resultados
):
    z = [
        r["zagreb1"]
        for r in resultados
    ]

    w = [
        r["wiener"]
        for r in resultados
    ]

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.scatter(
        w,
        z,
        s=9,
        alpha=0.35
    )

    ax.set_xlabel(
        r"Índice de Wiener $W$"
    )

    ax.set_ylabel(
        r"Primer índice de Zagreb $Z_1$"
    )

    ax.set_title(
        f"Relación Wiener–Zagreb, n={NUM_VERTICES}"
    )

    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_rango_entropia(
    resumen_df,
    campo,
    etiqueta,
    titulo
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        resumen_df[campo],
        resumen_df["entropia_min"],
        label="Entropía mínima"
    )

    ax.plot(
        resumen_df[campo],
        resumen_df["entropia_max"],
        label="Entropía máxima"
    )

    ax.fill_between(
        resumen_df[campo],
        resumen_df["entropia_min"],
        resumen_df["entropia_max"],
        alpha=0.2
    )

    ax.set_xlabel(etiqueta)

    ax.set_ylabel(
        "Entropía de von Neumann"
    )

    ax.set_title(titulo)

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_rango_secundario(
    resumen_df,
    campo,
    campo_secundario,
    etiqueta_x,
    etiqueta_y,
    titulo
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    minimo = (
        resumen_df[
            f"{campo_secundario}_min"
        ]
    )

    maximo = (
        resumen_df[
            f"{campo_secundario}_max"
        ]
    )

    ax.plot(
        resumen_df[campo],
        minimo,
        label="Mínimo"
    )

    ax.plot(
        resumen_df[campo],
        maximo,
        label="Máximo"
    )

    ax.fill_between(
        resumen_df[campo],
        minimo,
        maximo,
        alpha=0.2
    )

    ax.set_xlabel(etiqueta_x)
    ax.set_ylabel(etiqueta_y)
    ax.set_title(titulo)

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_discriminacion_global(
    resumen_df
):
    datos = resumen_df.sort_values(
        "porcentaje_discriminante"
    )

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.barh(
        datos["descriptor"],
        datos["porcentaje_discriminante"]
    )

    ax.set_xlabel(
        "Poder discriminante (%)"
    )

    ax.set_title(
        f"Poder discriminante, n={NUM_VERTICES}"
    )

    fig.tight_layout()

    return fig



def figura_perdidas_por_zagreb(
    resumen_zagreb_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    x = resumen_zagreb_df["zagreb1"]

    ax.plot(
        x,
        resumen_zagreb_df[
            "perdida_arbol_a_espectro"
        ],
        label=r"$N-Q$: árbol a espectro"
    )

    ax.plot(
        x,
        resumen_zagreb_df[
            "perdida_espectro_a_entropia"
        ],
        label=r"$Q-E$: espectro a entropía"
    )

    ax.plot(
        x,
        resumen_zagreb_df[
            "perdida_total_arbol_a_entropia"
        ],
        label=r"$N-E$: pérdida total"
    )

    ax.set_xlabel(
        r"Primer índice de Zagreb $Z_1$"
    )

    ax.set_ylabel(
        "Cantidad de distinciones perdidas"
    )

    ax.set_title(
        "Descomposición de la pérdida dentro de cada clase de Zagreb"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_tendencia_discriminacion(
    tendencia_df
):
    columnas = [
        ("D_S", "S"),
        ("D_Z1", "Z1"),
        ("D_W", "W"),
        ("D_Z1_W", "(Z1, W)"),
        ("D_S_Z1", "(S, Z1)"),
        ("D_S_W", "(S, W)"),
        ("D_S_Z1_W", "(S, Z1, W)"),
    ]

    fig, ax = plt.subplots(
        figsize=(11, 6.5)
    )

    for columna, etiqueta in columnas:
        ax.plot(
            tendencia_df["n"],
            100 * tendencia_df[columna],
            marker="o",
            label=etiqueta
        )

    ax.set_xlabel(
        "Número de vértices n"
    )

    ax.set_ylabel(
        "Poder discriminante (%)"
    )

    ax.set_title(
        "Evolución del poder discriminante"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_tendencia_correlacion_ZW(
    tendencia_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        tendencia_df["n"],
        tendencia_df[
            "correlacion_pearson_Z1_W"
        ],
        marker="o",
        label="Pearson"
    )

    ax.plot(
        tendencia_df["n"],
        tendencia_df[
            "correlacion_spearman_Z1_W"
        ],
        marker="s",
        label="Spearman"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )

    ax.set_ylabel(
        "Correlación"
    )

    ax.set_title(
        "Evolución de la relación Zagreb–Wiener"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_tendencia_resolucion_colisiones(
    tendencia_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        tendencia_df["n"],
        tendencia_df[
            "pct_colisiones_Z1_resueltas_por_W"
        ],
        marker="o",
        label="W resuelve colisiones de Z1"
    )

    ax.plot(
        tendencia_df["n"],
        tendencia_df[
            "pct_colisiones_W_resueltas_por_Z1"
        ],
        marker="s",
        label="Z1 resuelve colisiones de W"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )

    ax.set_ylabel(
        "Colisiones resueltas (%)"
    )

    ax.set_title(
        "Complementariedad entre Zagreb y Wiener"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_tendencia_rango_condicional(
    tendencia_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        tendencia_df["n"],
        100 * tendencia_df[
            "promedio_rango_relativo_W_dado_Z1"
        ],
        marker="o",
        label=r"Rango relativo de $W$ con $Z_1$ fijo"
    )

    ax.plot(
        tendencia_df["n"],
        100 * tendencia_df[
            "promedio_rango_relativo_Z1_dado_W"
        ],
        marker="s",
        label=r"Rango relativo de $Z_1$ con $W$ fijo"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )

    ax.set_ylabel(
        "Rango promedio relativo (%)"
    )

    ax.set_title(
        "Variación condicional relativa de Zagreb y Wiener"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig
def figura_wiener_vs_zagreb_ce(
    representantes_df
):
    """Representa espectros distintos dentro de clases coentrópicas."""
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    if representantes_df.empty:
        ax.text(
            0.5,
            0.5,
            "No hay clases coentrópicas con espectros distintos",
            ha="center",
            va="center",
            transform=ax.transAxes
        )

    else:
        ax.scatter(
            representantes_df["wiener"],
            representantes_df["zagreb1"],
            s=18,
            alpha=0.55
        )

    ax.set_xlabel(
        r"Índice de Wiener $W$"
    )

    ax.set_ylabel(
        r"Primer índice de Zagreb $Z_1$"
    )

    ax.set_title(
        "Wiener–Zagreb en espectros distintos "
        "dentro de clases coentrópicas"
    )

    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_discriminacion_espectral_ce(
    clases_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    if clases_df.empty:
        ax.text(
            0.5,
            0.5,
            "No hay clases coentrópicas con espectros distintos",
            ha="center",
            va="center",
            transform=ax.transAxes
        )

    else:
        datos = clases_df.sort_values(
            "entropia_representativa"
        )

        ax.plot(
            datos["entropia_representativa"],
            100 * datos["D_Z1_espectral"],
            marker="o",
            label="Z1"
        )

        ax.plot(
            datos["entropia_representativa"],
            100 * datos["D_W_espectral"],
            marker="s",
            label="W"
        )

        ax.plot(
            datos["entropia_representativa"],
            100 * datos["D_Z1_W_espectral"],
            marker="^",
            label="(Z1, W)"
        )

        ax.legend()

    ax.set_xlabel(
        "Entropía representativa de la clase"
    )

    ax.set_ylabel(
        "Poder discriminante espectral (%)"
    )

    ax.set_title(
        "Discriminación de espectros dentro de clases coentrópicas"
    )

    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_recuperacion_global_ce(
    resumen_df
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    etiquetas = [
        "Z1",
        "W",
        "(Z1, W)",
    ]

    fila = resumen_df.iloc[0]

    valores = [
        100 * fila[
            "recuperacion_perdida_Z1"
        ],
        100 * fila[
            "recuperacion_perdida_W"
        ],
        100 * fila[
            "recuperacion_perdida_Z1_W"
        ],
    ]

    ax.bar(
        etiquetas,
        valores
    )

    ax.set_ylabel(
        "Pérdida espectral recuperada (%)"
    )

    ax.set_title(
        "Información espectral recuperada "
        "dentro de las colisiones de entropía"
    )

    ax.set_ylim(
        0,
        105
    )

    fig.tight_layout()

    return fig


def figura_tendencia_recuperacion_ce(
    tendencia_ce_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        tendencia_ce_df["n"],
        100 * tendencia_ce_df[
            "CE_recuperacion_perdida_Z1"
        ],
        marker="o",
        label="Z1"
    )

    ax.plot(
        tendencia_ce_df["n"],
        100 * tendencia_ce_df[
            "CE_recuperacion_perdida_W"
        ],
        marker="s",
        label="W"
    )

    ax.plot(
        tendencia_ce_df["n"],
        100 * tendencia_ce_df[
            "CE_recuperacion_perdida_Z1_W"
        ],
        marker="^",
        label="(Z1, W)"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )

    ax.set_ylabel(
        "Pérdida espectral recuperada (%)"
    )

    ax.set_title(
        "Evolución de la recuperación dentro "
        "de clases coentrópicas no cospectrales"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_tendencia_clases_ce(
    tendencia_ce_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        tendencia_ce_df["n"],
        tendencia_ce_df[
            "CE_n_clases"
        ],
        marker="o",
        label="Clases de entropía"
    )

    ax.plot(
        tendencia_ce_df["n"],
        tendencia_ce_df[
            "CE_n_espectros_implicados"
        ],
        marker="s",
        label="Espectros implicados"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )

    ax.set_ylabel(
        "Cantidad"
    )

    ax.set_title(
        "Evolución de las colisiones espectrales de entropía"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_tendencia_separacion_pares_ce(
    tendencia_ce_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        tendencia_ce_df["n"],
        tendencia_ce_df[
            "CE_pct_pares_separados_Z1"
        ],
        marker="o",
        label="Z1"
    )

    ax.plot(
        tendencia_ce_df["n"],
        tendencia_ce_df[
            "CE_pct_pares_separados_W"
        ],
        marker="s",
        label="W"
    )

    ax.plot(
        tendencia_ce_df["n"],
        tendencia_ce_df[
            "CE_pct_pares_separados_Z1_W"
        ],
        marker="^",
        label="(Z1, W)"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )

    ax.set_ylabel(
        "Pares de espectros separados (%)"
    )

    ax.set_title(
        "Evolución de la separación de espectros "
        "dentro de clases coentrópicas"
    )

    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


## 7. Exportación de tablas y figuras

Se exportan los resúmenes en Excel, los análisis detallados en CSV y las figuras en PNG. Las bases SQLite originales permanecen sin modificaciones.


In [7]:

asegurar_carpetas()
tablas = {
    "Discriminacion": resumen_discriminacion_df,
    "Perdida_global": resumen_perdida_global_df,
    "Zagreb_fijo": resumen_zagreb_df,
    "Wiener_fijo": resumen_wiener_df,
    "Entropia_fija_muestra": entropia_fija_df,
    "Relacion_Zagreb_Wiener": relacion_zagreb_wiener_df,
    "CE_no_cospectrales_muestra": coentropicas_no_cospectrales_df,
    "CE_representantes_muestra": representantes_espectrales_ce_df,
    "CE_colisiones_muestra": colisiones_Z1_W_ce_df,
    "CE_resumen_global": resumen_coentropicas_no_cospectrales_df,
    "Tendencia_ordenes": tendencia_df,
    "Tendencia_CE": tendencia_coentropicas_no_cospectrales_df
}
for nombre,df in tablas.items():
    # Los análisis extensos se conservan en CSV.
    if nombre.endswith('_muestra'):
        continue
    df.to_csv(Path(CARPETA_TABLAS) / (nombre+'.csv'),index=False,encoding='utf-8-sig')
ruta_excel = exportar_resumenes_excel(tablas)

def guardar_figura(funcion, nombre):
    try:
        fig=funcion()
        fig.savefig(Path(CARPETA_IMAGENES)/nombre, dpi=300, bbox_inches='tight')
        if MOSTRAR_FIGURAS:
            display(fig)
        plt.close(fig)
        print('Figura:', nombre)
    except Exception as exc:
        # Registro de errores durante la exportación de figuras.
        print('Error al guardar',nombre,type(exc).__name__,exc)
        raise

guardar_figura(lambda: figura_entropia_vs_invariante(
    resultados_muestra, 'zagreb1',r'Primer índice de Zagreb $Z_1$',
    f'Entropía vs. Zagreb, n={NUM_VERTICES} (muestra visual)'),
    '01_entropia_vs_zagreb.png')
guardar_figura(lambda: figura_entropia_vs_invariante(
    resultados_muestra, 'wiener',r'Índice de Wiener $W$',
    f'Entropía vs. Wiener, n={NUM_VERTICES} (muestra visual)'),
    '02_entropia_vs_wiener.png')
guardar_figura(lambda: figura_wiener_vs_zagreb(resultados_muestra),
               '03_wiener_vs_zagreb.png')
guardar_figura(lambda: figura_rango_entropia(resumen_zagreb_df,'zagreb1',
    r'Primer índice de Zagreb $Z_1$','Rango de entropía con Zagreb fijo'),
    '04_rango_entropia_Z1_fijo.png')
guardar_figura(lambda: figura_rango_entropia(resumen_wiener_df,'wiener',
    r'Índice de Wiener $W$','Rango de entropía con Wiener fijo'),
    '05_rango_entropia_W_fijo.png')
guardar_figura(lambda: figura_rango_secundario(resumen_zagreb_df,'zagreb1','wiener',
    r'Primer índice de Zagreb $Z_1$',r'Índice de Wiener $W$',
    'Variación de Wiener con Zagreb fijo'), '06_rango_W_Z1_fijo.png')
guardar_figura(lambda: figura_rango_secundario(resumen_wiener_df,'wiener','zagreb1',
    r'Índice de Wiener $W$',r'Primer índice de Zagreb $Z_1$',
    'Variación de Zagreb con Wiener fijo'),'07_rango_Z1_W_fijo.png')
guardar_figura(lambda: figura_discriminacion_global(resumen_discriminacion_df),
               '08_poder_discriminante.png')
guardar_figura(lambda: figura_perdidas_por_zagreb(resumen_zagreb_df),
               '09_perdidas_por_zagreb.png')
guardar_figura(lambda: figura_tendencia_discriminacion(tendencia_para_graficos),
               '10_tendencia_discriminacion.png')
guardar_figura(lambda: figura_tendencia_correlacion_ZW(tendencia_para_graficos),
               '11_tendencia_correlacion_ZW.png')
guardar_figura(lambda: figura_tendencia_resolucion_colisiones(tendencia_para_graficos),
               '12_tendencia_resolucion_colisiones_ZW.png')
guardar_figura(lambda: figura_tendencia_rango_condicional(tendencia_para_graficos),
               '13_tendencia_rango_condicional_ZW.png')
guardar_figura(lambda: figura_wiener_vs_zagreb_ce(representantes_espectrales_ce_df),
               '14_wiener_vs_zagreb_CE_no_cospectrales.png')
guardar_figura(lambda: figura_discriminacion_espectral_ce(coentropicas_no_cospectrales_df),
               '15_discriminacion_espectral_CE.png')
guardar_figura(lambda: figura_recuperacion_global_ce(resumen_coentropicas_no_cospectrales_df),
               '16_recuperacion_global_CE.png')
guardar_figura(lambda: figura_tendencia_recuperacion_ce(tendencia_ce_para_graficos),
               '17_tendencia_recuperacion_CE.png')
guardar_figura(lambda: figura_tendencia_clases_ce(tendencia_ce_para_graficos),
               '18_tendencia_clases_CE.png')
guardar_figura(lambda: figura_tendencia_separacion_pares_ce(tendencia_ce_para_graficos),
               '19_tendencia_separacion_pares_CE.png')
print('Exportación terminada.')
print('Resultados:', Path(CARPETA_ANALISIS).resolve())
print('Tablas:', Path(CARPETA_TABLAS).resolve())
print('Imágenes:', Path(CARPETA_IMAGENES).resolve())
print('Excel:', ruta_excel.resolve())


Figura: 01_entropia_vs_zagreb.png
Figura: 02_entropia_vs_wiener.png
Figura: 03_wiener_vs_zagreb.png
Figura: 04_rango_entropia_Z1_fijo.png
Figura: 05_rango_entropia_W_fijo.png
Figura: 06_rango_W_Z1_fijo.png
Figura: 07_rango_Z1_W_fijo.png
Figura: 08_poder_discriminante.png
Figura: 09_perdidas_por_zagreb.png
Figura: 10_tendencia_discriminacion.png
Figura: 11_tendencia_correlacion_ZW.png
Figura: 12_tendencia_resolucion_colisiones_ZW.png
Figura: 13_tendencia_rango_condicional_ZW.png
Figura: 14_wiener_vs_zagreb_CE_no_cospectrales.png
Figura: 15_discriminacion_espectral_CE.png
Figura: 16_recuperacion_global_CE.png
Figura: 17_tendencia_recuperacion_CE.png
Figura: 18_tendencia_clases_CE.png
Figura: 19_tendencia_separacion_pares_CE.png
Exportación terminada.
Resultados: C:\Users\arnol\Downloads\analisis_desde_sqlite
Tablas: C:\Users\arnol\Downloads\analisis_desde_sqlite\orden_24\tablas
Imágenes: C:\Users\arnol\Downloads\analisis_desde_sqlite\orden_24\imagenes
Excel: C:\Users\arnol\Downloads\anal